# 🧠 Brain Tumor Classification — Complete Experiments (v5 — Enhanced)
**Experiment 1:** Centralized ResNet50 (30 epochs)
**Experiment 2:** Federated Learning IID — FedAvg (10 rounds × 3 local epochs = 30)
**Experiment 3:** Federated Learning Non-IID — FedAvg (10 rounds × 3 local epochs = 30)
**Experiment 4:** Federated Learning Non-IID — FedProx (proximal regularization, μ=0.01)
**Experiment 5:** Federated Learning IID — Focal Loss (γ=2, α=[2.0,1.2,1.0,1.0])
**Experiment 6:** Statistical Significance (5 seeds, paired t-test)
**Experiment 7:** Federated Learning IID — DP-SGD (Opacus, formal (ε,δ)-DP guarantee)

---
**Dataset:** Nickparvar (7,023 MRI images, 4 classes)
**Model:** ResNet50 (pretrained ImageNet)
**Fair Comparison:** Central 30 epochs = FL 10 rounds × 3 local epochs
**New in v5:** FedProx, Focal Loss for Glioma, Statistical Significance Testing, DP-SGD (Opacus)

**⚠️ Known limitations (see paper Limitations section):**
- Single dataset partitioned into virtual clients (not real multi-institutional data — TCGA/BraTS planned as future work)
- num_rounds kept at 10 for fair comparison with centralized baseline; longer-horizon convergence (20+ rounds) not yet tested


## ✅ Step 1: Install Libraries

In [ ]:
!pip install grad-cam opacus -q
print('✅ Libraries installed!')

## ✅ Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted!')

## ✅ Step 3: Verify Dataset

In [ ]:
import os

DATASET_PATH = '/content/drive/MyDrive/Nickparvar'
SAVE_DIR     = '/content/drive/MyDrive/BrainTumor_FL_Results_Final'
os.makedirs(SAVE_DIR, exist_ok=True)

print('📂 Dataset Structure:')
total = 0
for split in ['Training', 'Testing']:
    path = f'{DATASET_PATH}/{split}'
    for cls in sorted(os.listdir(path)):
        count = len(os.listdir(os.path.join(path, cls)))
        total += count
        print(f'  {split}/{cls}: {count} images')
print(f'\n✅ Total images: {total}')

## ✅ Step 4: Imports & Config

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
import numpy as np
import copy
import shutil
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import warnings
import random
warnings.filterwarnings('ignore')

# ✅ Global seed for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(42)
print('✅ Seed set to 42 — results will be reproducible!')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {DEVICE}')

# ════════════════════════════════════════════
#              CONFIGURATION
# Central 30 epochs = FL 10 rounds × 3 epochs
# ════════════════════════════════════════════
CONFIG = {
    # Federated Learning
    'num_clients'   : 3,
    'num_rounds'    : 10,
    'local_epochs'  : 3,

    # Centralized (fair comparison: 10 rounds x 3 epochs = 30)
    'central_epochs': 30,     # fair comparison: 10 rounds × 3 = 30

    # Model
    'batch_size'    : 32,
    'lr'            : 0.001,
    'num_classes'   : 4,
    'img_size'      : 224,
    'classes'       : ['glioma', 'meningioma', 'notumor', 'pituitary']
}

print('✅ Config ready!')
print(f'   Central Epochs : {CONFIG["central_epochs"]}')
print(f'   FL Rounds      : {CONFIG["num_rounds"]}')
print(f'   Local Epochs   : {CONFIG["local_epochs"]}')
print(f'   Total FL Steps : {CONFIG["num_rounds"]} × {CONFIG["local_epochs"]} = {CONFIG["num_rounds"]*CONFIG["local_epochs"]} ← equal to central')
print(f'   Batch Size     : {CONFIG["batch_size"]}')
print(f'   Learning Rate  : {CONFIG["lr"]}')

## ✅ Step 5: Data Transforms & Loading
> 🔧 CenterCrop fix — watermark/border Remove

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(CONFIG['img_size']),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(CONFIG['img_size']),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])

full_train_dataset = datasets.ImageFolder(f'{DATASET_PATH}/Training', transform=train_transform)
test_dataset       = datasets.ImageFolder(f'{DATASET_PATH}/Testing',  transform=test_transform)
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(42)

test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'],
                         shuffle=False, num_workers=2,
                         persistent_workers=True,
                         worker_init_fn=seed_worker,
                         generator=g)

print(f'✅ Training samples : {len(full_train_dataset)}')
print(f'✅ Test samples     : {len(test_dataset)}')
print(f'✅ Classes          : {full_train_dataset.classes}')

## ✅ Step 6: Common Functions

In [ ]:
# ── Model ──────────────────────────────────────────────────
def get_resnet50():
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    for name, param in model.named_parameters():
        if 'layer4' not in name and 'fc' not in name:
            param.requires_grad = False
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(model.fc.in_features, CONFIG['num_classes'])
    )
    return model

# ── Evaluate ───────────────────────────────────────────────
def evaluate(model, dataloader):
    model.eval()
    correct, total = 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total   += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return 100.*correct/total, all_preds, all_labels

# ── Confusion Matrix ───────────────────────────────────────
def plot_confusion(preds, labels, title, fname):
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=[c.upper() for c in CONFIG['classes']],
                yticklabels=[c.upper() for c in CONFIG['classes']])
    plt.title(title, fontsize=13, fontweight='bold')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Saved: {fname}')

# ── Classification Report ──────────────────────────────────
def print_report(preds, labels, title):
    print(f'\n{"="*55}')
    print(f'📊 {title}')
    print(f'{"="*55}')
    print(classification_report(labels, preds,
          target_names=[c.upper() for c in CONFIG['classes']]))

print('✅ Common functions ready!')

---
# 🔬 EXPERIMENT 1 — Centralized ResNet50
> Full dataset at a time — 30 epoch — no privacy

In [ ]:
set_seed(42)
print('='*55)
print('🚀 Experiment 1: Centralized Training')
print(f'   Epochs: {CONFIG["central_epochs"]}')
print('='*55)

central_model = get_resnet50().to(DEVICE)
full_loader = DataLoader(full_train_dataset, batch_size=CONFIG['batch_size'],
                        shuffle=True, num_workers=2,
                        persistent_workers=True,
                        worker_init_fn=seed_worker,
                        generator=g)
criterion     = nn.CrossEntropyLoss()
optimizer     = optim.Adam(filter(lambda p: p.requires_grad,
                           central_model.parameters()), lr=CONFIG['lr'])
central_accs  = []
best_central_acc = 0

for epoch in range(1, CONFIG['central_epochs']+1):
    central_model.train()
    correct, total, total_loss = 0, 0, 0
    for images, labels in full_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = central_model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total   += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    test_acc, _, _ = evaluate(central_model, test_loader)
    central_accs.append(test_acc)

    # ✅ Best model save
    if test_acc > best_central_acc:
        best_central_acc = test_acc
        torch.save(central_model.state_dict(), '/content/centralized_best_model.pth')
        best_marker = ' ⭐ Best!'
    else:
        best_marker = ''

    print(f'   Epoch {epoch:02d}/{CONFIG["central_epochs"]} | '
          f'Train Acc: {100.*correct/total:.2f}% | '
          f'Test Acc: {test_acc:.2f}%{best_marker}')

print(f'\n🎯 Centralized Final Accuracy : {central_accs[-1]:.2f}%')
print(f'🏆 Centralized Best Accuracy  : {best_central_acc:.2f}%')

In [ ]:
# Accuracy Curve
plt.figure(figsize=(9,5))
plt.plot(range(1, CONFIG['central_epochs']+1), central_accs, 'g-o',
         linewidth=2, markersize=5)
plt.title('Centralized ResNet50 — Test Accuracy per Epoch',
          fontsize=13, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Test Accuracy (%)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/central_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

# Report & Confusion Matrix
c_acc, c_preds, c_labels = evaluate(central_model, test_loader)
print_report(c_preds, c_labels, 'Centralized ResNet50 — Classification Report')
plot_confusion(c_preds, c_labels,
               'Confusion Matrix — Centralized ResNet50',
               '/content/central_confusion_matrix.png')

torch.save(central_model.state_dict(), '/content/centralized_model.pth')
print('✅ Centralized last-epoch model saved!')
print(f'🏆 Best epoch accuracy: {best_central_acc:.2f}% (saved as centralized_best_model.pth)')

---
# 🔬 EXPERIMENT 2 & 3 — FL Functions

In [ ]:
# ── IID Partition ──────────────────────────────────────────
def partition_iid(dataset, num_clients):
    indices = list(range(len(dataset)))
    np.random.shuffle(indices)
    splits  = np.array_split(indices, num_clients)
    loaders = []
    for i, idx in enumerate(splits):
        loader = DataLoader(Subset(dataset, idx),
                            batch_size=CONFIG['batch_size'],
                            shuffle=True, num_workers=2,
                            persistent_workers=True,
                            worker_init_fn=seed_worker,
                            generator=g)
        loaders.append(loader)
        print(f'  Client {i+1}: {len(idx)} samples')
    return loaders

# ── Non-IID Partition (FIXED) ──────────────────────────────
def partition_noniid(dataset, num_clients):
    """
    Realistic Non-IID:

    """
    labels = np.array([dataset[i][1] for i in range(len(dataset))])
    num_classes   = CONFIG['num_classes']
    class_indices = [np.where(labels == c)[0] for c in range(num_classes)]
    client_indices = [[] for _ in range(num_clients)]

    for c in range(num_classes):
        idx = class_indices[c].copy()
        np.random.shuffle(idx)
        # 70% → dominant client
        dominant = int(0.7 * len(idx))
        client_indices[c % num_clients].extend(idx[:dominant])

        remaining     = idx[dominant:]
        splits        = np.array_split(remaining, num_clients - 1)
        other_clients = [i for i in range(num_clients) if i != c % num_clients]
        for i, split in enumerate(splits):
            client_indices[other_clients[i]].extend(split)

    loaders = []
    for i, idx in enumerate(client_indices):
        idx = np.array(idx)
        np.random.shuffle(idx)
        client_labels = labels[idx]
        unique, counts = np.unique(client_labels, return_counts=True)
        dist = {CONFIG['classes'][u]: int(c) for u, c in zip(unique, counts)}
        loader = DataLoader(Subset(dataset, idx),
                            batch_size=CONFIG['batch_size'],
                            shuffle=True, num_workers=2, persistent_workers=True)
        loaders.append(loader)
        print(f'  Client {i+1}: {len(idx)} samples | {dist}')
    return loaders

# ── Local Train ────────────────────────────────────────────
def local_train(model, dataloader, epochs, lr):
    model.train()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad,
                           model.parameters()), lr=lr)
    correct, total, total_loss = 0, 0, 0
    for epoch in range(epochs):
        for images, labels in dataloader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total   += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return model.state_dict(), total_loss/total, 100.*correct/total

# ── FedAvg ────────────────────────────────────────────────
def fedavg(global_model, client_weights, client_sizes):
    total   = sum(client_sizes)
    g_state = copy.deepcopy(client_weights[0])
    for key in g_state.keys():
        g_state[key] = torch.zeros_like(g_state[key], dtype=torch.float32)
        for i, w in enumerate(client_weights):
            g_state[key] += (client_sizes[i]/total) * w[key].float()
    global_model.load_state_dict(g_state)
    return global_model

# ── FL Training Loop ──────────────────────────────────────
def fl_train(client_loaders, label):
    global_model = get_resnet50().to(DEVICE)
    client_sizes = [len(l.dataset) for l in client_loaders]
    round_accs    = []
    comm_costs    = []
    best_fl_acc   = 0
    best_fl_state = None
    trainable_params = sum(p.numel() for p in global_model.parameters()
                           if p.requires_grad)

    print(f'\n{"="*55}')
    print(f'🚀 Federated Training — {label}')
    print(f'   Rounds: {CONFIG["num_rounds"]} | Clients: {CONFIG["num_clients"]} | Local Epochs: {CONFIG["local_epochs"]}')
    print(f'{"="*55}')

    for rnd in range(1, CONFIG['num_rounds']+1):
        print(f'\n🔄 Round {rnd}/{CONFIG["num_rounds"]}')
        client_weights = []
        for cid, loader in enumerate(client_loaders):
            local_model = copy.deepcopy(global_model)
            weights, loss, acc = local_train(local_model, loader,
                                             CONFIG['local_epochs'], CONFIG['lr'])
            client_weights.append(weights)
            print(f'   Client {cid+1}: Loss={loss:.4f} | Train Acc={acc:.2f}%')

        global_model = fedavg(global_model, client_weights, client_sizes)
        test_acc, _, _ = evaluate(global_model, test_loader)
        round_accs.append(test_acc)
        # ✅ Upload + Download cost (×2 for bidirectional)
        cost_mb = (trainable_params * CONFIG['num_clients'] * 4 * 2) / (1024**2)
        comm_costs.append(cost_mb * rnd)

        # ✅ Best round model save
        if test_acc > best_fl_acc:
            best_fl_acc   = test_acc
            best_fl_state = copy.deepcopy(global_model.state_dict())
            best_marker   = ' ⭐ Best!'
        else:
            best_marker   = ''

        print(f'   ✅ Global Test Accuracy: {test_acc:.2f}%{best_marker} | '
              f'Cumulative Comm Cost: {comm_costs[-1]:.1f} MB')

    # Load best round weights before returning
    global_model.load_state_dict(best_fl_state)
    print(f'\n🎯 Final Accuracy ({label}): {round_accs[-1]:.2f}%')
    print(f'🏆 Best Accuracy  ({label}): {best_fl_acc:.2f}%')
    return global_model, round_accs, comm_costs, best_fl_acc

print('✅ FL functions ready!')

---
# 🔬 EXPERIMENT 2 — Federated Learning (IID)
> Data equally Devided by 3 — 10 rounds × 3 local epochs = 30

In [ ]:
set_seed(42)
print('Partitioning data (IID):')
iid_loaders = partition_iid(full_train_dataset, CONFIG['num_clients'])
fl_iid_model, iid_accs, iid_costs, best_iid_acc = fl_train(iid_loaders, 'IID')

In [ ]:
# IID Accuracy Curve
plt.figure(figsize=(8,5))
plt.plot(range(1, CONFIG['num_rounds']+1), iid_accs, 'b-o',
         linewidth=2, markersize=8)
plt.title('Federated Learning (IID) — Global Model Accuracy per Round',
          fontsize=13, fontweight='bold')
plt.xlabel('Communication Round')
plt.ylabel('Test Accuracy (%)')
plt.grid(True, alpha=0.3)
plt.xticks(range(1, CONFIG['num_rounds']+1))
for i, acc in enumerate(iid_accs):
    plt.annotate(f'{acc:.1f}%', (i+1, acc),
                 textcoords='offset points', xytext=(0,10), ha='center')
plt.tight_layout()
plt.savefig('/content/fl_iid_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

# IID Report & Confusion Matrix
iid_acc, iid_preds, iid_labels = evaluate(fl_iid_model, test_loader)
print_report(iid_preds, iid_labels, 'Federated IID — Classification Report')
plot_confusion(iid_preds, iid_labels,
               'Confusion Matrix — Federated ResNet50 (IID)',
               '/content/fl_iid_confusion_matrix.png')

torch.save(fl_iid_model.state_dict(), '/content/fl_iid_model.pth')
print('✅ FL IID model saved!')

---
# 🔬 EXPERIMENT 3 — Federated Learning (Non-IID Fixed)
>

In [ ]:
set_seed(42)
print('Partitioning data (Non-IID Fixed):')
noniid_loaders = partition_noniid(full_train_dataset, CONFIG['num_clients'])
fl_noniid_model, noniid_accs, noniid_costs, best_noniid_acc = fl_train(noniid_loaders, 'Non-IID')

In [ ]:
# Non-IID Accuracy Curve
plt.figure(figsize=(8,5))
plt.plot(range(1, CONFIG['num_rounds']+1), noniid_accs, 'r-o',
         linewidth=2, markersize=8)
plt.title('Federated Learning (Non-IID) — Global Model Accuracy per Round',
          fontsize=13, fontweight='bold')
plt.xlabel('Communication Round')
plt.ylabel('Test Accuracy (%)')
plt.grid(True, alpha=0.3)
plt.xticks(range(1, CONFIG['num_rounds']+1))
for i, acc in enumerate(noniid_accs):
    plt.annotate(f'{acc:.1f}%', (i+1, acc),
                 textcoords='offset points', xytext=(0,10), ha='center')
plt.tight_layout()
plt.savefig('/content/fl_noniid_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

# Non-IID Report & Confusion Matrix
noniid_acc, noniid_preds, noniid_labels = evaluate(fl_noniid_model, test_loader)
print_report(noniid_preds, noniid_labels, 'Federated Non-IID — Classification Report')
plot_confusion(noniid_preds, noniid_labels,
               'Confusion Matrix — Federated ResNet50 (Non-IID)',
               '/content/fl_noniid_confusion_matrix.png')

torch.save(fl_noniid_model.state_dict(), '/content/fl_noniid_model.pth')
print('✅ FL Non-IID model saved!')

---
# 🔬 EXPERIMENT 4 — FedProx (Non-IID)
> Proximal regularization term prevents client drift in heterogeneous data

In [ ]:
# ══════════════════════════════════════════════════════════
# 🆕 EXPERIMENT 4 — FedProx Functions
# ══════════════════════════════════════════════════════════

def local_train_fedprox(model, global_state, dataloader, epochs, lr, mu=0.01):
    """FedProx local training with proximal regularization term."""
    model.train()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad,
                           model.parameters()), lr=lr)
    correct, total, total_loss = 0, 0, 0
    for epoch in range(epochs):
        for images, labels in dataloader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)

            # ✅ Proximal term: penalise deviation from global model
            prox = 0.0
            for name, param in model.named_parameters():
                if param.requires_grad:
                    prox += ((param - global_state[name].to(DEVICE)) ** 2).sum()
            loss = loss + (mu / 2) * prox

            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total   += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return model.state_dict(), total_loss / total, 100. * correct / total


def fl_train_fedprox(client_loaders, label, mu=0.01):
    """Federated training using FedProx aggregation."""
    global_model = get_resnet50().to(DEVICE)
    client_sizes = [len(l.dataset) for l in client_loaders]
    round_accs   = []
    comm_costs   = []
    best_fl_acc  = 0
    best_fl_state = None
    trainable_params = sum(p.numel() for p in global_model.parameters()
                           if p.requires_grad)

    print(f'\n{"="*60}')
    print(f'🚀 FedProx Training — {label}  (mu={mu})')
    print(f'   Rounds: {CONFIG["num_rounds"]} | Clients: {CONFIG["num_clients"]} | Local Epochs: {CONFIG["local_epochs"]}')
    print(f'{"="*60}')

    for rnd in range(1, CONFIG['num_rounds'] + 1):
        print(f'\n🔄 Round {rnd}/{CONFIG["num_rounds"]}')
        global_state   = copy.deepcopy(global_model.state_dict())
        client_weights = []
        for cid, loader in enumerate(client_loaders):
            local_model = copy.deepcopy(global_model)
            weights, loss, acc = local_train_fedprox(
                local_model, global_state, loader,
                CONFIG['local_epochs'], CONFIG['lr'], mu=mu)
            client_weights.append(weights)
            print(f'   Client {cid+1}: Loss={loss:.4f} | Train Acc={acc:.2f}%')

        global_model = fedavg(global_model, client_weights, client_sizes)
        test_acc, _, _ = evaluate(global_model, test_loader)
        round_accs.append(test_acc)
        cost_mb = (trainable_params * CONFIG['num_clients'] * 4 * 2) / (1024 ** 2)
        comm_costs.append(cost_mb * rnd)

        if test_acc > best_fl_acc:
            best_fl_acc   = test_acc
            best_fl_state = copy.deepcopy(global_model.state_dict())
            best_marker   = ' ⭐ Best!'
        else:
            best_marker   = ''

        print(f'   ✅ Global Test Accuracy: {test_acc:.2f}%{best_marker} | '
              f'Cumulative Comm Cost: {comm_costs[-1]:.1f} MB')

    global_model.load_state_dict(best_fl_state)
    print(f'\n🎯 Final Accuracy ({label}): {round_accs[-1]:.2f}%')
    print(f'🏆 Best Accuracy  ({label}): {best_fl_acc:.2f}%')
    return global_model, round_accs, comm_costs, best_fl_acc

print('✅ FedProx functions ready!')


In [ ]:
# ══════════════════════════════════════════════════════════
# 🔬 EXPERIMENT 4 — FedProx Non-IID
# ══════════════════════════════════════════════════════════
set_seed(42)
print('Partitioning data (Non-IID for FedProx):')
noniid_loaders_prox = partition_noniid(full_train_dataset, CONFIG['num_clients'])
fl_fedprox_model, fedprox_accs, fedprox_costs, best_fedprox_acc = \
    fl_train_fedprox(noniid_loaders_prox, 'Non-IID FedProx', mu=0.01)


In [ ]:
# FedProx vs FedAvg Non-IID comparison plot
plt.figure(figsize=(9, 5))
plt.plot(range(1, CONFIG['num_rounds']+1), noniid_accs,   'r-s',
         linewidth=2, markersize=8, label='FedAvg Non-IID')
plt.plot(range(1, CONFIG['num_rounds']+1), fedprox_accs,  'm-^',
         linewidth=2, markersize=8, label='FedProx Non-IID')
plt.axhline(y=central_accs[-1], color='g', linestyle='--', linewidth=2,
            label=f'Centralized ({central_accs[-1]:.1f}%)')
plt.title('FedAvg vs FedProx — Non-IID Setting',
          fontsize=13, fontweight='bold')
plt.xlabel('Communication Round')
plt.ylabel('Test Accuracy (%)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(range(1, CONFIG['num_rounds']+1))
for i, acc in enumerate(fedprox_accs):
    plt.annotate(f'{acc:.1f}%', (i+1, acc),
                 textcoords='offset points', xytext=(0, 10), ha='center', color='purple')
plt.tight_layout()
plt.savefig('/content/fedprox_vs_fedavg.png', dpi=150, bbox_inches='tight')
plt.show()

fedprox_acc_val, fedprox_preds, fedprox_labels = evaluate(fl_fedprox_model, test_loader)
print_report(fedprox_preds, fedprox_labels, 'FedProx Non-IID — Classification Report')
plot_confusion(fedprox_preds, fedprox_labels,
               'Confusion Matrix — FedProx (Non-IID)',
               '/content/fedprox_confusion_matrix.png')

torch.save(fl_fedprox_model.state_dict(), '/content/fl_fedprox_model.pth')
print(f'\n✅ FedProx improvement over FedAvg Non-IID: '
      f'{best_fedprox_acc - best_noniid_acc:+.2f}%')


---
# 🔬 EXPERIMENT 5 — Focal Loss for Glioma Improvement
> CrossEntropyLoss → Focal Loss to fix class imbalance and hard example mining for Glioma


In [ ]:
# ══════════════════════════════════════════════════════════
# 🆕 Focal Loss — addresses Glioma's hard-to-classify cases
# ══════════════════════════════════════════════════════════
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None):
        super().__init__()
        self.gamma = gamma
        # Higher weight for Glioma (class 0) and Meningioma (class 1)
        self.alpha = alpha if alpha is not None else torch.tensor(
            [2.0, 1.2, 1.0, 1.0])  # glioma gets 2× weight

    def forward(self, inputs, targets):
        ce_loss = nn.CrossEntropyLoss(reduction='none')(inputs, targets)
        pt      = torch.exp(-ce_loss)
        alpha_t = self.alpha.to(inputs.device)[targets]
        focal   = alpha_t * (1 - pt) ** self.gamma * ce_loss
        return focal.mean()


def local_train_focal(model, dataloader, epochs, lr):
    model.train()
    criterion = FocalLoss(gamma=2.0)
    optimizer = optim.Adam(filter(lambda p: p.requires_grad,
                           model.parameters()), lr=lr)
    correct, total, total_loss = 0, 0, 0
    for epoch in range(epochs):
        for images, labels in dataloader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total   += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return model.state_dict(), total_loss / total, 100. * correct / total


def fl_train_focal(client_loaders, label):
    global_model = get_resnet50().to(DEVICE)
    client_sizes = [len(l.dataset) for l in client_loaders]
    round_accs   = []
    best_fl_acc  = 0
    best_fl_state = None

    print(f'\n{"="*60}')
    print(f'🚀 Focal Loss FL Training — {label}')
    print(f'{"="*60}')

    for rnd in range(1, CONFIG['num_rounds'] + 1):
        print(f'\n🔄 Round {rnd}/{CONFIG["num_rounds"]}')
        client_weights = []
        for cid, loader in enumerate(client_loaders):
            local_model = copy.deepcopy(global_model)
            weights, loss, acc = local_train_focal(
                local_model, loader, CONFIG['local_epochs'], CONFIG['lr'])
            client_weights.append(weights)
            print(f'   Client {cid+1}: Loss={loss:.4f} | Train Acc={acc:.2f}%')

        global_model = fedavg(global_model, client_weights, client_sizes)
        test_acc, _, _ = evaluate(global_model, test_loader)
        round_accs.append(test_acc)

        if test_acc > best_fl_acc:
            best_fl_acc   = test_acc
            best_fl_state = copy.deepcopy(global_model.state_dict())
            best_marker   = ' ⭐ Best!'
        else:
            best_marker   = ''
        print(f'   ✅ Global Test Accuracy: {test_acc:.2f}%{best_marker}')

    global_model.load_state_dict(best_fl_state)
    print(f'\n🏆 Best Accuracy (Focal {label}): {best_fl_acc:.2f}%')
    return global_model, round_accs, best_fl_acc


set_seed(42)
print('Running FL IID with Focal Loss...')
iid_loaders_focal = partition_iid(full_train_dataset, CONFIG['num_clients'])
fl_focal_model, focal_accs, best_focal_acc = fl_train_focal(iid_loaders_focal, 'IID')

focal_acc_val, focal_preds, focal_labels = evaluate(fl_focal_model, test_loader)
print_report(focal_preds, focal_labels, 'FL IID Focal Loss — Classification Report')
plot_confusion(focal_preds, focal_labels,
               'Confusion Matrix — FL IID Focal Loss',
               '/content/focal_confusion_matrix.png')

from sklearn.metrics import f1_score
orig_glioma_f1  = f1_score(iid_labels,   iid_preds,   average=None)[0]
focal_glioma_f1 = f1_score(focal_labels, focal_preds, average=None)[0]
print(f'\n📊 Glioma F1 — FL IID CrossEntropy : {orig_glioma_f1:.4f}')
print(f'📊 Glioma F1 — FL IID Focal Loss   : {focal_glioma_f1:.4f}')
print(f'📈 Improvement: {focal_glioma_f1 - orig_glioma_f1:+.4f}')
torch.save(fl_focal_model.state_dict(), '/content/fl_focal_model.pth')


---
# 📊 Statistical Significance Analysis
> 5 independent runs with different seeds → Mean ± Std + paired t-test


In [ ]:
from scipy import stats
import pandas as pd

SEEDS = [42, 123, 256, 789, 999]

def run_single_experiment(seed):
    set_seed(seed)
    # ── Centralized ──
    model_c  = get_resnet50().to(DEVICE)
    loader_c = DataLoader(full_train_dataset, batch_size=CONFIG['batch_size'],
                          shuffle=True, num_workers=2, persistent_workers=True)
    crit = nn.CrossEntropyLoss()
    opt  = optim.Adam(filter(lambda p: p.requires_grad, model_c.parameters()),
                      lr=CONFIG['lr'])
    for _ in range(CONFIG['central_epochs']):
        model_c.train()
        for imgs, lbls in loader_c:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            opt.zero_grad()
            loss = crit(model_c(imgs), lbls)
            loss.backward(); opt.step()
    acc_c, _, _ = evaluate(model_c, test_loader)

    # ── FL IID ──
    set_seed(seed)
    iid_l = partition_iid(full_train_dataset, CONFIG['num_clients'])
    _, _, _, acc_iid = fl_train(iid_l, f'IID-seed{seed}')

    return acc_c, acc_iid

print('🔬 Running 5-seed statistical significance experiment...')
print('   ⚠️  This takes ~10× normal training time. Please wait.\n')

central_runs, iid_runs = [], []
for seed in SEEDS:
    print(f'\n── Seed {seed} ──────────────────────────────')
    ac, ai = run_single_experiment(seed)
    central_runs.append(ac)
    iid_runs.append(ai)
    print(f'   Centralized: {ac:.2f}% | FL IID: {ai:.2f}%')

# ── Summary stats ──
c_mean, c_std = np.mean(central_runs), np.std(central_runs)
i_mean, i_std = np.mean(iid_runs),     np.std(iid_runs)
t_stat, p_val = stats.ttest_rel(iid_runs, central_runs)

print('\n' + '='*60)
print('📊 STATISTICAL SIGNIFICANCE RESULTS')
print('='*60)
print(f'  Centralized : {c_mean:.2f}% ± {c_std:.2f}%')
print(f'  FL IID      : {i_mean:.2f}% ± {i_std:.2f}%')
print(f'  Paired t-test: t={t_stat:.4f}, p={p_val:.4f}')
if p_val < 0.05:
    print('  ✅ Difference is statistically SIGNIFICANT (p < 0.05)')
else:
    print('  ℹ️  Difference is NOT significant (p ≥ 0.05) — models are equivalent')
print('='*60)

# Save results to CSV for paper
results_df = pd.DataFrame({
    'Seed':        SEEDS,
    'Centralized': central_runs,
    'FL_IID':      iid_runs
})
results_df.to_csv('/content/statistical_results.csv', index=False)
print('\n✅ Results saved to /content/statistical_results.csv')
print('\n📝 Paper-ready line:')
print(f'   Centralized ResNet50: {c_mean:.2f}% ± {c_std:.2f}%')
print(f'   Federated IID:        {i_mean:.2f}% ± {i_std:.2f}%')
print(f'   (paired t-test, p={p_val:.3f})')


---
# 🔬 EXPERIMENT 7 — Federated Learning with Differential Privacy (DP-SGD)
> Formal privacy guarantee using Opacus — gradient clipping + calibrated noise injection.
> This addresses the "privacy is only informal (no raw data sharing)" critique by providing
> a mathematically provable (ε, δ)-DP guarantee per client.


In [ ]:
# ══════════════════════════════════════════════════════════
# 🆕 EXPERIMENT 7 — DP-SGD via Opacus
# ══════════════════════════════════════════════════════════
from opacus import PrivacyEngine
from opacus.utils.batch_memory_manager import BatchMemoryManager

# DP Configuration — standard values used in DP-SGD literature
DP_CONFIG = {
    'noise_multiplier': 1.0,     # σ — controls how much noise is added
    'max_grad_norm':    1.0,     # C — per-sample gradient clipping bound
    'target_delta':      1e-5,    # δ — probability of privacy failure
}

def local_train_dp(model, dataloader, epochs, lr, noise_multiplier=1.0, max_grad_norm=1.0):
    """
    Local client training with DP-SGD (Opacus).
    Returns: state_dict, avg_loss, accuracy, epsilon (privacy budget spent)
    """
    model.train()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad,
                           model.parameters()), lr=lr)

    privacy_engine = PrivacyEngine()
    model_dp, optimizer_dp, dataloader_dp = privacy_engine.make_private(
        module=model,
        optimizer=optimizer,
        data_loader=dataloader,
        noise_multiplier=noise_multiplier,
        max_grad_norm=max_grad_norm,
    )

    correct, total, total_loss = 0, 0, 0
    for epoch in range(epochs):
        for images, labels in dataloader_dp:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer_dp.zero_grad()
            outputs = model_dp(images)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer_dp.step()
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total   += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    epsilon = privacy_engine.get_epsilon(delta=DP_CONFIG['target_delta'])

    # Opacus wraps the model in GradSampleModule — extract clean state_dict
    clean_state = model_dp._module.state_dict()
    return clean_state, total_loss / total, 100. * correct / total, epsilon


def fl_train_dp(client_loaders, label, noise_multiplier=1.0, max_grad_norm=1.0):
    """Federated training where each client trains with DP-SGD."""
    global_model = get_resnet50().to(DEVICE)
    client_sizes = [len(l.dataset) for l in client_loaders]
    round_accs    = []
    epsilons      = []
    best_fl_acc   = 0
    best_fl_state = None

    print(f'\n{"="*60}')
    print(f'🔐 DP-SGD Federated Training — {label}')
    print(f'   noise_multiplier={noise_multiplier} | max_grad_norm={max_grad_norm} | δ={DP_CONFIG["target_delta"]}')
    print(f'{"="*60}')

    for rnd in range(1, CONFIG['num_rounds'] + 1):
        print(f'\n🔄 Round {rnd}/{CONFIG["num_rounds"]}')
        client_weights = []
        round_epsilons = []
        for cid, loader in enumerate(client_loaders):
            local_model = copy.deepcopy(global_model)
            weights, loss, acc, eps = local_train_dp(
                local_model, loader, CONFIG['local_epochs'], CONFIG['lr'],
                noise_multiplier=noise_multiplier, max_grad_norm=max_grad_norm)
            client_weights.append(weights)
            round_epsilons.append(eps)
            print(f'   Client {cid+1}: Loss={loss:.4f} | Train Acc={acc:.2f}% | ε={eps:.2f}')

        global_model = fedavg(global_model, client_weights, client_sizes)
        test_acc, _, _ = evaluate(global_model, test_loader)
        round_accs.append(test_acc)
        epsilons.append(max(round_epsilons))  # worst-case client epsilon for this round

        if test_acc > best_fl_acc:
            best_fl_acc   = test_acc
            best_fl_state = copy.deepcopy(global_model.state_dict())
            best_marker   = ' ⭐ Best!'
        else:
            best_marker   = ''

        print(f'   ✅ Global Test Accuracy: {test_acc:.2f}%{best_marker} | Cumulative ε: {epsilons[-1]:.2f}')

    global_model.load_state_dict(best_fl_state)
    print(f'\n🎯 Final Accuracy ({label}): {round_accs[-1]:.2f}%')
    print(f'🏆 Best Accuracy  ({label}): {best_fl_acc:.2f}%')
    print(f'🔐 Final Privacy Budget (ε, δ): ({epsilons[-1]:.2f}, {DP_CONFIG["target_delta"]})')
    return global_model, round_accs, epsilons, best_fl_acc


set_seed(42)
print('Partitioning data (IID) for DP-SGD experiment:')
iid_loaders_dp = partition_iid(full_train_dataset, CONFIG['num_clients'])
fl_dp_model, dp_accs, dp_epsilons, best_dp_acc = fl_train_dp(
    iid_loaders_dp, 'IID + DP-SGD',
    noise_multiplier=DP_CONFIG['noise_multiplier'],
    max_grad_norm=DP_CONFIG['max_grad_norm'])


In [ ]:
# ── DP-SGD Results: Accuracy vs Privacy Budget ──
dp_acc_val, dp_preds, dp_labels = evaluate(fl_dp_model, test_loader)
print_report(dp_preds, dp_labels, 'Federated IID + DP-SGD — Classification Report')
plot_confusion(dp_preds, dp_labels,
               'Confusion Matrix — Federated ResNet50 (IID + DP-SGD)',
               '/content/fl_dp_confusion_matrix.png')

# Accuracy curve with privacy budget annotated
fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(range(1, CONFIG['num_rounds']+1), dp_accs, 'c-o', linewidth=2, markersize=8, label='DP-SGD Accuracy')
ax1.plot(range(1, CONFIG['num_rounds']+1), iid_accs, 'b--', linewidth=1.5, alpha=0.6, label='FedAvg IID (no DP)')
ax1.set_xlabel('Communication Round')
ax1.set_ylabel('Test Accuracy (%)')
ax1.set_title('Privacy-Utility Trade-off — FL IID with DP-SGD', fontsize=13, fontweight='bold')
ax1.legend(loc='lower left')
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(range(1, CONFIG['num_rounds']+1), dp_epsilons, 'r:^', linewidth=2, markersize=6, label='Privacy Budget (ε)')
ax2.set_ylabel('Cumulative Privacy Budget (ε)', color='red')
ax2.tick_params(axis='y', labelcolor='red')
ax2.legend(loc='lower right')

plt.tight_layout()
plt.savefig('/content/dp_privacy_utility_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()

torch.save(fl_dp_model.state_dict(), '/content/fl_dp_model.pth')

print('\n' + '='*65)
print('📊 PRIVACY-UTILITY TRADE-OFF SUMMARY')
print('='*65)
print(f'  FedAvg IID (no formal privacy) : {best_iid_acc:.2f}%')
print(f'  FedAvg IID + DP-SGD             : {best_dp_acc:.2f}%')
print(f'  Accuracy cost of formal privacy : {best_iid_acc-best_dp_acc:.2f}%')
print(f'  Final (ε, δ)-DP guarantee       : ({dp_epsilons[-1]:.2f}, {DP_CONFIG["target_delta"]})')
print('='*65)
print('\n📝 Paper-ready line:')
print(f'   "With DP-SGD (noise_multiplier={DP_CONFIG["noise_multiplier"]}, max_grad_norm={DP_CONFIG["max_grad_norm"]}),')
print(f'   the federated model achieves {best_dp_acc:.2f}% accuracy under a formal')
print(f'   ({dp_epsilons[-1]:.2f}, {DP_CONFIG["target_delta"]})-differential privacy guarantee,')
print(f'   compared to {best_iid_acc:.2f}% without DP — an accuracy cost of only')
print(f'   {best_iid_acc-best_dp_acc:.2f} percentage points for provable per-client privacy."')


---
# 📊 Final Comparison — All Experiment are at a time

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

def get_metrics(preds, labels):
    p, r, f, _ = precision_recall_fscore_support(labels, preds, average=None)
    acc = 100.*sum(pp==ll for pp,ll in zip(preds,labels))/len(labels)
    return p, r, f, acc

cp,  cr,  cf,  c_acc  = get_metrics(c_preds,      c_labels)
ip,  ir,  if_, i_acc  = get_metrics(iid_preds,    iid_labels)
n_prec, n_rec, n_f1, n_acc = get_metrics(noniid_preds, noniid_labels)

print('\n' + '='*75)
print('📊 FINAL COMPARISON TABLE')
print('='*75)
print(f'{"Method":<22} {"Accuracy":>9} {"Glioma F1":>10} {"Mening F1":>10} {"NoTumor F1":>11} {"Pituit F1":>10}')
print('-'*75)
print(f'{"Centralized":<22} {c_acc:>8.2f}% {cf[0]:>10.3f} {cf[1]:>10.3f} {cf[2]:>11.3f} {cf[3]:>10.3f}')
print(f'{"Federated IID":<22} {i_acc:>8.2f}% {if_[0]:>10.3f} {if_[1]:>10.3f} {if_[2]:>11.3f} {if_[3]:>10.3f}')
print(f'{"Federated Non-IID":<22} {n_acc:>8.2f}% {n_f1[0]:>10.3f} {n_f1[1]:>10.3f} {n_f1[2]:>11.3f} {n_f1[3]:>10.3f}')
print('='*75)
print(f'\n💡 Privacy-Performance Trade-off:')
print(f'   Centralized → FL IID gap   : {c_acc-i_acc:.2f}%')
print(f'   FL IID → FL Non-IID gap    : {i_acc-n_acc:.2f}%')

---
## 📊 Updated Full Comparison — All Experiments

In [ ]:
# ══════════════════════════════════════════════════════════
# 📊 UPDATED Final Comparison Table — All 6 Experiments
# ══════════════════════════════════════════════════════════
from sklearn.metrics import precision_recall_fscore_support

def get_metrics(preds, labels):
    p, r, f, _ = precision_recall_fscore_support(labels, preds, average=None)
    acc = 100. * sum(pp == ll for pp, ll in zip(preds, labels)) / len(labels)
    return p, r, f, acc

cp,     cr,  cf,  c_acc   = get_metrics(c_preds,       c_labels)
ip,     ir,  if_, i_acc   = get_metrics(iid_preds,     iid_labels)
n_prec, n_rec, n_f1, n_acc = get_metrics(noniid_preds, noniid_labels)
pp,     pr,  pf,  p_acc   = get_metrics(fedprox_preds, fedprox_labels)
fp,     fr,  ff,  f_acc   = get_metrics(focal_preds,   focal_labels)
dpp,    dpr, dpf, dp_acc_ = get_metrics(dp_preds,       dp_labels)

print('\n' + '='*92)
print('📊 COMPLETE COMPARISON TABLE — ALL EXPERIMENTS')
print('='*92)
header = f'{"Method":<28} {"Accuracy":>9} {"Glioma F1":>10} {"Mening F1":>10} {"NoTumor F1":>11} {"Pituit F1":>10}'
print(header)
print('-'*92)
rows = [
    ('Centralized (CE)',         c_acc,   cf[0],   cf[1],   cf[2],   cf[3]),
    ('Federated IID (FedAvg)',   i_acc,  if_[0],  if_[1],  if_[2],  if_[3]),
    ('Federated Non-IID (FedAvg)', n_acc, n_f1[0], n_f1[1], n_f1[2], n_f1[3]),
    ('FedProx Non-IID',          p_acc,   pf[0],   pf[1],   pf[2],   pf[3]),
    ('FL IID + Focal Loss',      f_acc,   ff[0],   ff[1],   ff[2],   ff[3]),
    ('FL IID + DP-SGD',          dp_acc_, dpf[0],  dpf[1],  dpf[2],  dpf[3]),
]
for name, acc, g, m, n, pit in rows:
    print(f'{name:<28} {acc:>8.2f}% {g:>10.3f} {m:>10.3f} {n:>11.3f} {pit:>10.3f}')
print('='*92)
print(f'\n🏆 Best Glioma F1            : {max(cf[0],if_[0],n_f1[0],pf[0],ff[0],dpf[0]):.3f}')
print(f'🏆 Best Overall Accuracy     : {max(c_acc,i_acc,n_acc,p_acc,f_acc,dp_acc_):.2f}%')
print(f'🔒 FedProx vs FedAvg Non-IID : {p_acc-n_acc:+.2f}% improvement')
print(f'🎯 Focal vs CE Glioma F1     : {ff[0]-if_[0]:+.3f} improvement')
print(f'🔐 Privacy cost of DP-SGD    : {i_acc-dp_acc_:.2f}% (ε={dp_epsilons[-1]:.2f}, δ={DP_CONFIG["target_delta"]})')

# Save full results to CSV for paper tables
import pandas as pd
full_results = pd.DataFrame(rows, columns=['Method','Accuracy','Glioma_F1','Meningioma_F1','NoTumor_F1','Pituitary_F1'])
full_results.to_csv('/content/full_comparison_results.csv', index=False)
print('\n✅ Full results saved to /content/full_comparison_results.csv')


In [ ]:
# Comparison Accuracy Curve
plt.figure(figsize=(9,5))
plt.plot(range(1, CONFIG['num_rounds']+1), iid_accs,    'b-o',
         linewidth=2, markersize=7, label='FL IID')
plt.plot(range(1, CONFIG['num_rounds']+1), noniid_accs, 'r-s',
         linewidth=2, markersize=7, label='FL Non-IID (Fixed)')
plt.axhline(y=central_accs[-1], color='g', linestyle='--', linewidth=2,
            label=f'Centralized ({central_accs[-1]:.1f}%)')
plt.title('Accuracy — Centralized vs FL IID vs FL Non-IID',
          fontsize=13, fontweight='bold')
plt.xlabel('Communication Round')
plt.ylabel('Test Accuracy (%)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(range(1, CONFIG['num_rounds']+1))
plt.tight_layout()
plt.savefig('/content/comparison_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Comparison plot saved!')

In [ ]:
# Communication Cost Plot
plt.figure(figsize=(8,5))
plt.plot(range(1, CONFIG['num_rounds']+1), iid_costs,    'b-o',
         linewidth=2, markersize=7, label='FL IID')
plt.plot(range(1, CONFIG['num_rounds']+1), noniid_costs, 'r-s',
         linewidth=2, markersize=7, label='FL Non-IID')
plt.title('Communication Cost per Round (Cumulative MB)',
          fontsize=13, fontweight='bold')
plt.xlabel('Communication Round')
plt.ylabel('Cumulative Cost (MB)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(range(1, CONFIG['num_rounds']+1))
plt.tight_layout()
plt.savefig('/content/communication_cost.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Communication cost plot saved!')

## ✅ Grad-CAM — FL IID Global Model

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

cam = GradCAM(model=fl_iid_model, target_layers=[fl_iid_model.layer4[-1]])

class_samples = {i: None for i in range(CONFIG['num_classes'])}
for images, labels in test_loader:
    for img, label in zip(images, labels):
        label = label.item()
        if class_samples[label] is None:
            class_samples[label] = img
    if all(v is not None for v in class_samples.values()):
        break

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Grad-CAM — Federated ResNet50 (IID)\n'
             '(Top: Original MRI | Bottom: Grad-CAM Heatmap)',
             fontsize=13, fontweight='bold')

mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])

for idx, (class_idx, img_tensor) in enumerate(class_samples.items()):
    input_tensor  = img_tensor.unsqueeze(0).to(DEVICE)
    grayscale_cam = cam(input_tensor=input_tensor,
                        targets=[ClassifierOutputTarget(class_idx)])[0]
    img_np  = np.clip(std * img_tensor.permute(1,2,0).numpy() + mean,
                      0, 1).astype(np.float32)
    cam_img = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)

    axes[0][idx].imshow(img_np)
    axes[0][idx].set_title(f'{CONFIG["classes"][class_idx].upper()}\n(Original)', fontsize=10)
    axes[0][idx].axis('off')
    axes[1][idx].imshow(cam_img)
    axes[1][idx].set_title(f'{CONFIG["classes"][class_idx].upper()}\n(Grad-CAM)', fontsize=10)
    axes[1][idx].axis('off')

plt.tight_layout()
plt.savefig('/content/gradcam_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Grad-CAM saved!')

---
## 📋 Final Summary

| Experiment | Method | Setting | Key Feature |
|---|---|---|---|
| 1 | Centralized ResNet50 | Full data, no privacy | Baseline |
| 2 | Federated IID | 3 clients, equal split | Privacy |
| 3 | Federated Non-IID | 3 clients, 70/30 split | Privacy + Heterogeneity |
| 4 | FedProx Non-IID | 3 clients, 70/30 split | Proximal regularization |
| 5 | FL IID + Focal Loss | 3 clients, equal split | Glioma improvement |
| 6 | Statistical Significance | 5 seeds | Mean ± Std + t-test |
| 7 | FL IID + DP-SGD | 3 clients, equal split | Formal (ε,δ)-Differential Privacy |

### Output Files
```
├── Models        : centralized, fl_iid, fl_noniid, fl_fedprox, fl_focal, fl_dp (.pth)
├── Accuracy      : central + iid + noniid + fedprox + focal + dp curves
├── Confusion     : 6 × confusion matrix
├── Comparison    : fedprox_vs_fedavg, dp_privacy_utility_tradeoff, full comparison
├── Comm Cost     : communication cost curve
├── Statistical   : statistical_results.csv
├── Full Results  : full_comparison_results.csv
└── Grad-CAM      : 4 class heatmaps (FL IID global model)
```

### 📝 Paper-Ready Results Template
```
Table: Overall Accuracy Comparison
Method                    | Best Acc | Final Acc | Glioma F1 | Privacy
---------------------------|----------|-----------|-----------|----------------
Centralized ResNet50      | X.XX%    | X.XX%     | 0.XX      | None
Federated IID (FedAvg)    | X.XX%    | X.XX%     | 0.XX      | Informal (no data sharing)
Federated Non-IID         | X.XX%    | X.XX%     | 0.XX      | Informal
FedProx Non-IID           | X.XX%    | X.XX%     | 0.XX      | Informal + heterogeneity-robust
FL IID + Focal Loss       | X.XX%    | X.XX%     | 0.XX ↑    | Informal
FL IID + DP-SGD           | X.XX%    | X.XX%     | 0.XX      | Formal (ε=X.XX, δ=1e-5)
```

### Known Limitations (to state explicitly in paper)
1. **Simulated FL**: single dataset partitioned into virtual clients, not genuine multi-institutional data.
2. **Round budget**: capped at 10 rounds to match the centralized 30-epoch baseline; longer training horizons (20+ rounds) were not evaluated due to compute constraints.
3. **DP-SGD trade-off**: per-sample gradient clipping (Opacus) changes effective batch dynamics slightly vs. standard SGD — accuracy comparisons should be read alongside the reported ε.
